# 13 — Astra vision + GeoAI: explain a change map

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/13_astra_vision_change.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/13_astra_vision_change.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `2590c6a7f0ca7d4891e2913cddc3696761fb231de423f888e91de4edf5076984`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 13 — Astra vision + GeoAI: explain a change map

[Launch in JupyterLite](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=13_astra_vision_change.ipynb)

Build paired synthetic scenes, derive a change mask, evaluate it against known truth, and package a labeled comparison image for Astra. **Time:** about 25 minutes. **Prerequisites:** arrays and lab 03.

**Runtime:** browser Python for the analysis; hosted GeoLibre for maps; optional CPython runner for Astra. The default run makes no model calls. The first run needs network access for packages, GeoLibre and interactive Plotly scripts. Static plots and local analysis do not depend on data APIs.

**Learning cycle:** predict → run → compare views → explain → test a different assumption.

## Start here: the question behind the code

**How can an image explanation stay tied to reproducible evidence?**

**What you will learn:** compare → mask → evaluate → explain. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Image-analysis foundation with optional vision-model interpretation; the default RGB threshold is a deterministic rule.

**Why this is useful:** Known synthetic truth makes mistakes visible before the method is transferred to real Earth imagery.

**Where the work happens:** Synthetic images, masks, area and error counts run in Pyodide. A separate optional runner can send the exported image and metrics to a model.

![Measure change before explaining it. Calculate a per-pixel RGB difference between aligned invented scenes. Exclude the known synthetic cloud and apply an explicit threshold. Compare prediction with known change using error counts and overlap. Use computed area and a labeled figure to support, then check, an explanation.](assets/learning/13/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: Measure change before explaining it — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/13/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/13/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/13/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=13_astra_vision_change.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/13_astra_vision_change.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/13/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Compare (0–8 s):** Calculate a per-pixel RGB difference between aligned invented scenes. Code landmark: `distance = norm(after-before)`.

2. **Mask (8–16 s):** Exclude the known synthetic cloud and apply an explicit threshold. Code landmark: `predicted = (distance > threshold) & valid`.

3. **Evaluate (16–24 s):** Compare prediction with known change using error counts and overlap. Code landmark: `iou = tp / (tp + fp + fn)`.

4. **Explain (24–32 s):** Use computed area and a labeled figure to support, then check, an explanation. Code landmark: `evidence['detected_area_m2']`.


## See the connections

![Astra vision: compare, measure, challenge: Images: Aligned before and after teaching chips; GeoAI pattern: Threshold change and evaluate known truth; Astra: Read labeled images with numeric evidence; Check: Appearance alone cannot establish cause](assets/13_mindmap.svg)

### Code guide 1 — Prepare the browser toolbox

**Read the Python:** `import` makes a package available to Python. `sys.platform` identifies the runtime; the indented `if` block runs only in Pyodide. `await` waits for package loading. `LiteMap as Map` gives the browser-compatible map class a shorter name. Installing a package and importing its functions are separate steps.

**Look for:** Imports complete before you run the next cell. The first run can take longer while packages download.

**Pause and try:** Find the alias `Map`. If a later cell says a name is undefined, rerun this setup cell first.


In [ ]:
import sys
if sys.platform == "emscripten":
    from pyodide import loadPackage
    await loadPackage(["numpy", "pandas", "matplotlib", "scipy", "pillow"])
    import micropip
    await micropip.install(["geolibre==3.0.0", "plotly==6.5.0"])
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from geolibre_lite import LiteMap as Map
from cognition import style_plots, COLORS, save_json
from astra_lab import make_request, response_text
style_plots()

## 1. A controlled scene makes mistakes measurable

These are **synthetic RGB chips**, not satellite imagery. One rectangle changes color, a bright patch imitates cloud, and random noise perturbs the second image. Both images share a 64 × 64 local grid with 20 m cells. Row zero is south. The anchored coordinates are illustrative, not a surveyed raster geotransform.

### Code guide 2 — Create a scene with known change

**Read the Python:** A fixed random seed makes the noise repeatable. Boolean arrays identify the invented changed rectangle and cloud. `np.linalg.norm(..., axis=2)` measures RGB difference per pixel; a threshold and validity mask define detected change. True/false positive/negative counts compare detection with known synthetic truth; IoU measures overlap.

**Look for:** A change mask and overlap score for an easy controlled example with an oracle cloud mask.

**Pause and try:** Predict what happens to false positives if you stop excluding cloud pixels.


In [ ]:
from cognition import grid_features, figure_data_url
rng = np.random.default_rng(42)
n = 64
yy, xx = np.mgrid[:n,:n]
before = np.empty((n,n,3), dtype=float)
before[:] = [.18,.48,.25]
before[:,28:34] = [.12,.25,.52]
before += rng.normal(0,.025,before.shape)
truth = (xx>=40) & (xx<54) & (yy>=12) & (yy<30)
cloud = (xx-16)**2 + (yy-45)**2 < 8**2
after = before.copy()
after[truth] = [.65,.42,.27]
after[cloud] = [.92,.92,.92]
after += rng.normal(0,.02,after.shape)
before, after = np.clip(before,0,1), np.clip(after,0,1)
valid = ~cloud  # oracle mask known because we generated the fixture
distance = np.linalg.norm(after-before, axis=2)
threshold = .22
predicted = (distance > threshold) & valid
tp = int((predicted & truth & valid).sum())
fp = int((predicted & ~truth & valid).sum())
fn = int((~predicted & truth & valid).sum())
tn = int((~predicted & ~truth & valid).sum())
iou = tp/(tp+fp+fn) if tp+fp+fn else 1.0
print("Valid pixels:", int(valid.sum()), "IoU:", round(iou,3))

### Code guide 3 — Explain success and mistakes visually

**Read the Python:** The four image panels reuse one extent and orientation. Histograms compare known changed and unchanged pixels. The confusion matrix puts truth on rows and prediction on columns, with text counts so it remains readable without color.

**Look for:** Before/after images, difference, detection, distributions and error counts.

**Pause and try:** Point to the false-positive square and explain the kind of mistake it counts.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), layout="constrained")
axes = axes.ravel()
extent = (0,1.28,0,1.28)
axes[0].imshow(before, origin="lower", extent=extent)
axes[1].imshow(after, origin="lower", extent=extent)
change_colors = axes[2].imshow(np.ma.masked_where(~valid,distance), origin="lower", extent=extent, cmap="magma", vmin=0,vmax=1)
axes[3].imshow(predicted, origin="lower", extent=extent, cmap="Blues", vmin=0,vmax=1)
for ax,title in zip(axes,["A: synthetic before","B: synthetic after","C: RGB change (cloud masked)","D: threshold change mask"]):
    ax.set(title=title,xlabel="Local east (km)",ylabel="Local north (km)")
fig.colorbar(change_colors,ax=axes[2],label="RGB Euclidean distance")
from matplotlib.patches import Patch
axes[3].legend(handles=[Patch(color="#08306b",label="Detected change"),Patch(color="#f7fbff",label="No detected change")],loc="upper left",fontsize=8)
image_evidence = figure_data_url(fig)
plt.show()
fig, axes = plt.subplots(1,2,figsize=(10,4),layout="constrained")
axes[0].hist(distance[valid & ~truth], bins=30, alpha=.7, label="Known unchanged")
axes[0].hist(distance[valid & truth], bins=30, alpha=.7, label="Known changed")
axes[0].axvline(threshold,color="#dc2626",ls="--")
axes[0].set(xlabel="Euclidean distance in RGB [0,1]",ylabel="Pixels",title="Threshold separates this easy fixture")
axes[0].legend()
matrix = np.array([[tn,fp],[fn,tp]])
display(pd.DataFrame(matrix, index=['Known: no change', 'Known: change'], columns=['Predicted: no change', 'Predicted: change']))
axes[1].imshow(matrix,cmap="Blues")
axes[1].set_xticks([0,1],["No change","Change"])
axes[1].set_yticks([0,1],["No change","Change"])
axes[1].set(xlabel="Predicted",ylabel="Known synthetic truth",title="Cloud pixels excluded from evaluation")
for r in range(2):
    for c in range(2):
        axes[1].text(c,r,str(matrix[r,c]),ha="center",va="center",color="white" if matrix[r,c]>matrix.max()/2 else "black")
plt.show()

## 2. Map the same cells and their area

Each cell covers exactly 400 m² in the **synthetic local grid**. Count cells there before applying the approximate longitude/latitude display transform. Do not calculate area in square degrees.

### Code guide 4 — Convert detected pixels into mapped area

**Read the Python:** `grid_features` turns the boolean result into synthetic 20 m grid cells. A list comprehension keeps only value 1. Multiplying the detected-cell count by 20² calculates the area in the local teaching grid.

**Look for:** Mapped change polygons, an area value in m² and a GeoJSON export.

**Pause and try:** Why must you multiply by pixel area instead of by pixel width?


In [ ]:
change_fc = grid_features(predicted.astype(float), field="changed", cell_m=20)
change_fc['features'] = [f for f in change_fc['features'] if f['properties']['changed']==1]
change_map = Map(center=(-122.293,37.906),zoom=14,height="480px")
if change_fc['features']:
    change_map.add_geojson(change_fc, name="Synthetic detected change",fillColor="#d97706",strokeWidth=0)
display(change_map)
area_m2 = int(predicted.sum()) * 20**2
print("Synthetic detected area (m²):",area_m2)
save_json("exports/synthetic_change.geojson",change_fc)

## 3. Give Astra an image and computed evidence

The vision request asks for observations and missing checks, not invented spectral measurements or causes. The numeric evidence carries pixel size, masks, confusion counts and IoU. An image-only answer could mistake the bright patch for meaningful land change.

### Code guide 5 — Give the optional model measured evidence

**Read the Python:** The evidence includes counts, units, mask size, threshold and source. The request combines these values with a labeled PNG. The prompt asks for observations and missing checks; numerical measurements still come from the code.

**Look for:** A saved multimodal request containing one figure; no live answer is generated by this cell.

**Pause and try:** Which parts of an explanation could the image support, and which require dates, calibration or local knowledge?


In [ ]:
evidence = {"evidence_id":"change_metrics", "source":"synthetic RGB fixture, seed 42",
            "pixel_size_m":20, "orientation":"north up; row 0 south in arrays", "threshold":threshold,
            "valid_pixels":int(valid.sum()), "masked_cloud_pixels":int(cloud.sum()),
            "tp":tp,"fp":fp,"fn":fn,"tn":tn,"iou":iou,"detected_area_m2":area_m2}
request = make_request("Compare panels A-D. Cite change_metrics. Distinguish observed color changes from possible causes. Explain why this easy synthetic result does not validate a real EO model.",
                       evidence, images=[image_evidence])
save_json("exports/vision_change_request.json",request)
print("Request contains one labeled PNG and computed metrics; not sent.")

## Optional: run this request with GPT-6 Astra

All calculations above run locally. The exported request has **not been sent** and any example plan is **hand-authored**, not a recorded model response. To obtain a live response:

1. Download the generated request from JupyterLite's file browser (right-click → Download).
2. In a full CPython terminal in this repository, set `OPENAI_API_KEY` in that terminal's environment. Never put the key in a notebook, output, or the static site.
3. Preview with `python scripts/run_astra.py PATH_TO_REQUEST.json`. Send deliberately with `--send --output astra_response.json`. This uses your API account and can incur charges.
4. Upload `astra_response.json` to the notebook folder, set the import path in the next cell, and rerun it. Inspect its evidence claims before using them.

For notebook 15, also download `accessibility_evidence.json` and pass `--evidence PATH_TO_EVIDENCE.json`. The runner only dispatches the one allowlisted local tool, with a bounded number of rounds.

The static JupyterLite site does not host this runner or proxy requests. Model access depends on your API project. Image reading is an aid to interpretation; geometry, distances and metrics come from code and explicit metadata.

### Code guide 6 — Inspect an optional model response

**Read the Python:** `None` means no response file was supplied. Only a real path enters the `if` branch; `json.loads` converts saved JSON text into Python objects. The `else` branch keeps the default lesson usable without an API call. Follow the preceding export/import instructions for live use.

**Look for:** The default run prints an offline message. An imported answer is evidence to inspect, not a new measurement.

**Pause and try:** Write your own explanation first, then check any imported response against the computed values and units.


In [ ]:
RESPONSE_PATH = None
if RESPONSE_PATH:
    report = json.loads(response_text(json.loads(Path(RESPONSE_PATH).read_text())))
    display(report)
else:
    print("No model explanation generated. Use the evidence to write your own interpretation first.")

## 4. Stress the explanation

Set `valid = np.ones_like(cloud)` and rerun from the mask calculation to see false positives from cloud. Raise the threshold and observe missed changes. Shift one image by one pixel to test registration error.

**Real-data handoff:** use STAC IDs, acquisition times, calibrated bands, a documented CRS/transform, cloud masks and independent labels. In full GeoAI, compare a trained change detector with this simple baseline; return its georeferenced predictions to GeoLibre. Astra can review that evidence, but an attractive explanation does not substitute for validation.

## Data & software citations

- Teaching fixtures: deterministic synthetic arrays and networks authored in this repository. Values, scenes, facilities and outcomes are invented; coordinates only anchor the display. No actual people, environmental measurements, or service areas are represented.
- [GeoLibre](https://geolibre.app/) and [GeoLibre map controls](https://geolibre.app/user-guide/map-controls/); `geolibre==3.0.0` project builders through `geolibre_lite.LiteMap`.
- [GeoAI](https://opengeoai.org/): full Python/GPU continuation for imagery models. These browser labs implement workflow patterns; they do not install or run `geoai-py`.
- [GPT-6 Astra model](https://developers.openai.com/api/docs/models/gpt-6-astra), [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs), [image inputs](https://developers.openai.com/api/docs/guides/images-vision), and [function calling](https://developers.openai.com/api/docs/guides/function-calling). API examples checked 2026-10-02.
- [Pyodide packages](https://pyodide.org/en/stable/usage/packages-in-pyodide.html), [Matplotlib](https://matplotlib.org/stable/), [Plotly Python](https://plotly.com/python/).
- GeoLibre basemap tiles retain their provider attribution, including [OpenStreetMap contributors](https://www.openstreetmap.org/copyright) where applicable.

See `DATA_SOURCES.md` for the source register and runtime boundaries.

## Make sense of the result

![All bright changes must be new buildings! The cloud also changed appearance. Check the mask. I will measure errors before explaining causes.](assets/learning/13/cartoon.svg)

**Cartoon transcript:** All bright changes must be new buildings! → The cloud also changed appearance. Check the mask. → I will measure errors before explaining causes.

**Explain it to a colleague:** Why does excellent accuracy here not validate a real satellite model?

<details><summary>Check your reasoning</summary>

The scene and cloud truth are deliberately easy and known. Real imagery adds misalignment, seasonality, sensor and labeling uncertainty.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
